# StyleMatch V1 Corpus Builder (Colab)\n\nThis notebook builds the initial English-only literary and rhetorical corpus. It does not generate imitation text. It only downloads source texts from approved or reviewable public sources and records metadata.

## Policy\n\n- No LLM-generated author/source text.\n- Campaign material excluded from v1 rhetorical corpus.\n- Literary: Project Gutenberg/public-domain prose.\n- Rhetorical: official/public-domain public rhetoric first.\n- Modern public figures with unclear transcript rights stay in `candidate_public_figures.csv`, not training data.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\nfrom pathlib import Path\nROOT = Path('/content/drive/MyDrive/stylematch_v1')\n(ROOT / 'data/source_registry').mkdir(parents=True, exist_ok=True)\n(ROOT / 'scripts').mkdir(parents=True, exist_ok=True)\nROOT

In [ ]:
!pip -q install requests beautifulsoup4 pandas tqdm

In [ ]:
import csv, json, re, time\nfrom pathlib import Path\nimport requests\nimport pandas as pd\nfrom tqdm.auto import tqdm\n\nSESSION = requests.Session()\nSESSION.headers.update({'User-Agent': 'StyleMatch corpus builder; academic prototype'})\n\ndef slug(value):\n    value = value.lower().replace('&', 'and')\n    value = re.sub(r'[^a-z0-9]+', '_', value)\n    return value.strip('_')\n\ndef write_csv(path, rows):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        return\n    with path.open('w', newline='', encoding='utf-8') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))\n        writer.writeheader()\n        writer.writerows(rows)\n\ndef clean_gutenberg(text):\n    text = text.replace('\\r\\n', '\\n').replace('\\r', '\\n')\n    start = re.search(r'\\*\\*\\* START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK .*?\\*\\*\\*', text, flags=re.I | re.S)\n    end = re.search(r'\\*\\*\\* END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK .*?\\*\\*\\*', text, flags=re.I | re.S)\n    if start:\n        text = text[start.end():]\n    if end:\n        text = text[:end.start()]\n    return re.sub(r'\\n{3,}', '\\n\\n', text).strip()\n\ndef chunk_words(text, min_words=75, max_words=150):\n    paragraphs = [p.strip() for p in re.split(r'\\n\\s*\\n', text) if p.strip()]\n    chunks, buf = [], []\n    for paragraph in paragraphs:\n        units = re.split(r'(?<=[.!?])\\s+', paragraph) if len(paragraph.split()) > max_words * 2 else [paragraph]\n        for unit in units:\n            buf.extend(unit.split())\n            if len(buf) >= min_words:\n                chunks.append(' '.join(buf[:max_words]))\n                buf = buf[max_words:]\n    if len(buf) >= min_words:\n        chunks.append(' '.join(buf))\n    return chunks

In [ ]:
literary_authors = [\n    'Jane Austen', 'Charles Dickens', 'George Eliot', 'Thomas Hardy', 'Mark Twain',\n    'Henry James', 'Edith Wharton', 'Herman Melville', 'Nathaniel Hawthorne', 'Edgar Allan Poe',\n    'Washington Irving', 'Louisa May Alcott', 'Kate Chopin', 'Willa Cather', 'Jack London',\n    'Stephen Crane', 'Charlotte Bronte', 'Emily Bronte', 'Anne Bronte', 'Mary Wollstonecraft Shelley',\n    'Robert Louis Stevenson', 'Joseph Conrad', 'H. G. Wells', 'Frances Hodgson Burnett',\n    'L. M. Montgomery', 'Elizabeth Cleghorn Gaskell', 'Anthony Trollope', 'Wilkie Collins',\n    'Arthur Conan Doyle', 'George MacDonald'\n]\n\nrhetorical_speakers = [\n    'George Washington', 'John Adams', 'Thomas Jefferson', 'James Madison', 'James Monroe',\n    'John Quincy Adams', 'Andrew Jackson', 'Martin Van Buren', 'John Tyler', 'James K. Polk',\n    'Zachary Taylor', 'Millard Fillmore', 'Franklin Pierce', 'James Buchanan', 'Abraham Lincoln',\n    'Andrew Johnson', 'Ulysses S. Grant', 'Rutherford B. Hayes', 'James A. Garfield',\n    'Chester A. Arthur', 'Grover Cleveland', 'Benjamin Harrison', 'William McKinley',\n    'Theodore Roosevelt', 'William Howard Taft', 'Woodrow Wilson', 'Warren G. Harding',\n    'Calvin Coolidge', 'Herbert Hoover', 'Franklin D. Roosevelt', 'Harry S. Truman',\n    'Dwight D. Eisenhower', 'John F. Kennedy', 'Lyndon B. Johnson', 'Richard Nixon',\n    'Gerald Ford', 'Jimmy Carter', 'Ronald Reagan', 'George H. W. Bush', 'Bill Clinton',\n    'George W. Bush', 'Barack Obama', 'Donald Trump', 'Joe Biden'\n]\n\ncandidates = [\n    ('Winston Churchill', 'UK/public speeches exist; copyright and edition rights need review'),\n    ('Martin Luther King Jr.', 'speeches are estate-controlled/copyrighted; do not ingest without license'),\n    ('Adolf Hitler', 'English translations/editions have translator and edition rights; do not ingest now'),\n    ('Pope Francis', 'Vatican/publication copyright terms need review'),\n    ('Pope Benedict XVI', 'Vatican/publication copyright terms need review'),\n    ('Pope John Paul II', 'Vatican/publication copyright terms need review'),\n    ('Bill Gates', 'public talks usually copyrighted by venue/publisher/Gates Notes'),\n    ('Elon Musk', 'interview/transcript sources usually copyrighted'),\n    ('Steve Jobs', 'Stanford/Apple text rights unclear for training redistribution'),\n    ('Nelson Mandela', 'reuse rights vary by source and edition'),\n    ('Margaret Thatcher', 'public archive exists; license needs review'),\n    ('Tony Blair', 'official/public speeches exist; Crown/copyright status needs review'),\n]\n\nwrite_csv(ROOT / 'data/source_registry/literary_authors.csv', [\n    {'author': a, 'status': 'approved', 'source_family': 'Project Gutenberg', 'notes': 'public-domain prose; verify each book header'}\n    for a in literary_authors\n])\nwrite_csv(ROOT / 'data/source_registry/rhetorical_speakers.csv', [\n    {'speaker': s, 'status': 'approved', 'source_family': 'official presidential public documents / APP / Project Gutenberg where available', 'notes': 'no campaign material'}\n    for s in rhetorical_speakers\n])\nwrite_csv(ROOT / 'data/source_registry/candidate_public_figures.csv', [\n    {'name': name, 'corpus': 'rhetorical', 'status': 'candidate', 'reason_not_ingested': reason}\n    for name, reason in candidates\n])

In [ ]:
def gutendex_books(name):\n    r = SESSION.get('https://gutendex.com/books/', params={'languages': 'en', 'search': name}, timeout=30)\n    r.raise_for_status()\n    return r.json().get('results', [])\n\ndef author_match(book, name):\n    wanted = set(re.findall(r'[a-z]+', name.lower()))\n    for author in book.get('authors', []):\n        got = set(re.findall(r'[a-z]+', author.get('name', '').lower()))\n        if wanted and wanted.issubset(got):\n            return True\n    return False\n\ndef text_url(book):\n    for k, v in book.get('formats', {}).items():\n        if k.startswith('text/plain') and not v.endswith('.zip'):\n            return v\n    return None\n\ndef download_text(url, max_seconds=120):\n    with SESSION.get(url, stream=True, timeout=30) as r:\n        r.raise_for_status()\n        chunks, started = [], time.monotonic()\n        for chunk in r.iter_content(chunk_size=65536):\n            if time.monotonic() - started > max_seconds:\n                raise TimeoutError(url)\n            if chunk:\n                chunks.append(chunk)\n    return b''.join(chunks).decode('utf-8', errors='replace')\n\ndef fetch_gutenberg_corpus(names, corpus, max_works=1):\n    rows = []\n    for name in tqdm(names):\n        found = 0\n        for book in gutendex_books(name):\n            if found >= max_works:\n                break\n            if not author_match(book, name):\n                continue\n            url = text_url(book)\n            if not url:\n                continue\n            try:\n                text = clean_gutenberg(download_text(url))\n            except Exception as e:\n                print('skip', name, book.get('id'), e)\n                continue\n            if len(text.split()) < 1000:\n                continue\n            out_dir = ROOT / f'data/{corpus}/raw' / slug(name)\n            out_dir.mkdir(parents=True, exist_ok=True)\n            text_path = out_dir / f'gutenberg_{book["id"]}.txt'\n            text_path.write_text(text, encoding='utf-8')\n            row = {\n                'corpus': corpus, 'author_or_speaker': name, 'title': book.get('title', ''),\n                'gutenberg_id': str(book['id']), 'source_url': url,\n                'license': 'Project Gutenberg; verify book header and U.S. public-domain status before redistribution',\n                'language': 'en', 'raw_text_path': str(text_path.relative_to(ROOT))\n            }\n            (out_dir / f'gutenberg_{book["id"]}.json').write_text(json.dumps(row, indent=2), encoding='utf-8')\n            rows.append(row)\n            found += 1\n    write_csv(ROOT / f'data/{corpus}/meta/sources.csv', rows)\n    return pd.DataFrame(rows)

In [ ]:
literary_sources = fetch_gutenberg_corpus(literary_authors, 'literary', max_works=1)\nliterary_sources.groupby('author_or_speaker').size().sort_values(ascending=False).head(), literary_sources.shape

Rhetorical fetching should be source-specific. Do not scrape campaign websites or copyrighted transcripts. Start with official presidential public documents; add APP/API fetcher only after source review.

In [ ]:
def chunk_corpus(corpus):\n    source_path = ROOT / f'data/{corpus}/meta/sources.csv'\n    if not source_path.exists():\n        print('missing', source_path)\n        return pd.DataFrame()\n    out_dir = ROOT / f'data/{corpus}/chunks'\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows = []\n    for source in pd.read_csv(source_path).to_dict('records'):\n        text = (ROOT / source['raw_text_path']).read_text(encoding='utf-8', errors='replace')\n        for i, chunk in enumerate(chunk_words(text), start=1):\n            chunk_id = f'{corpus}_{source["gutenberg_id"]}_{i:04d}'\n            chunk_path = out_dir / f'{chunk_id}.txt'\n            chunk_path.write_text(chunk, encoding='utf-8')\n            rows.append({\n                'chunk_id': chunk_id, 'corpus': corpus, 'author_or_speaker': source['author_or_speaker'],\n                'title': source['title'], 'source_id': source['gutenberg_id'], 'language': 'en',\n                'word_count': len(chunk.split()), 'chunk_path': str(chunk_path.relative_to(ROOT))\n            })\n    write_csv(ROOT / f'data/{corpus}/meta/chunks.csv', rows)\n    return pd.DataFrame(rows)\n\nliterary_chunks = chunk_corpus('literary')\nliterary_chunks.groupby('author_or_speaker').size().describe(), literary_chunks.shape